# Phase 04J: Teacher B → S2 Sequence-KD Training Readiness

Phase 04J khóa production engine trước full run seed 42:

1. xác minh Phase 04I và toàn bộ pseudo-caption corpus 113.287 ảnh;
2. ghép một human caption luân phiên với một Teacher-B pseudo caption/ảnh;
3. dựng lại S2 MobileViT-S và shared Transformer decoder từ checkpoint gốc;
4. tối ưu `0.5 × CE_human + 0.5 × CE_teacher_sequence`;
5. giữ nguyên AdamW, warmup-cosine, FP16 GradScaler và batch 32/64;
6. kiểm tra Validation-only, Beam-3 và COCO metrics;
7. chạy pilot 4 optimizer-step rồi xóa trọng số pilot;
8. kiểm tra atomic checkpoint/resume và authorize Phase 04K seed-42 full run.

Không nạp Teacher B trong Phase 04J: tri thức Teacher đã được cố định trong
pseudo corpus Phase 04I. Held-out Test luôn `SEALED`.

## Kaggle inputs bắt buộc

- Phase 01 formal output;
- Phase 02A shared Student bundle;
- Phase 04I full pseudo-caption corpus;
- MS COCO 2014 images;
- offline `pycocoevalcap==1.2` wheel/dataset.

## Runtime

- GPU T4 hoặc mạnh hơn; Internet Off.
- Chạy Cell 1 → Cell 7 theo thứ tự.


## Cell 1 — Authorization, environment và integrity

Xác minh marker COMPLETE/AUTHORIZED của Phase 04I, 69 artifact và SHA-256,
Phase 1/Phase 2A. Chưa đọc caption, ảnh hoặc checkpoint payload.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_b_s2_sequence_kd_training_readiness"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, LOG_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04J bắt buộc GPU. Trong Kaggle chọn GPU T4 hoặc mạnh hơn, "
        "khởi động lại session rồi chạy lại từ Cell 1."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4I_COMPLETE = exactly_one(
    "PHASE4_TEACHER_B_S2_PSEUDO_CORPUS_COMPLETE.json"
)
PHASE4J_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "TRAINING_READINESS_AUTHORIZED.json"
)
PHASE2_READY = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

phase4i_complete = read_json(PHASE4I_COMPLETE)
phase4j_authorized = read_json(PHASE4J_AUTHORIZED)
phase2_ready = read_json(PHASE2_READY)
phase1_ready = read_json(PHASE1_READY)

assert phase4i_complete["status"] == "complete"
assert phase4i_complete["experiment_id"] == "P4B_B_S2_SEQKD"
assert int(phase4i_complete["images"]) == 113287
assert int(phase4i_complete["shards"]) == 56
assert phase4j_authorized["status"] == "authorized"
assert phase4j_authorized["experiment_id"] == "P4B_B_S2_SEQKD"
assert phase4j_authorized["next_phase"] == (
    "Phase 04J S2 Sequence-KD training readiness"
)
assert int(phase4j_authorized["pseudo_corpus"]["rows"]) == 113287
assert phase2_ready["status"] == "ready"
assert phase1_ready["status"] == "ready"
for marker in [phase4i_complete, phase4j_authorized]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

PHASE4I_ROOT = PHASE4I_COMPLETE.parent
authorization_ref = phase4i_complete["next_authorization"]
assert PHASE4J_AUTHORIZED == resolve_inside(
    PHASE4I_ROOT, authorization_ref["relative_path"]
)
assert sha256_file(PHASE4J_AUTHORIZED) == authorization_ref["sha256"]

PSEUDO_CORPUS = resolve_inside(
    PHASE4I_ROOT,
    phase4j_authorized["pseudo_corpus"]["relative_path"],
)
assert sha256_file(PSEUDO_CORPUS) == (
    phase4j_authorized["pseudo_corpus"]["sha256"]
)
assert sha256_file(PSEUDO_CORPUS) == (
    phase4i_complete["pseudo_corpus"]["sha256"]
)

PHASE4I_MANIFEST = resolve_inside(
    PHASE4I_ROOT, "manifests/phase4i_manifest.json"
)
manifest_payload = read_json(PHASE4I_MANIFEST)
assert int(manifest_payload["artifact_count"]) == 69
assert len(manifest_payload["artifacts"]) == 69
expected_sum_lines = []
for row in manifest_payload["artifacts"]:
    artifact = resolve_inside(PHASE4I_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
    expected_sum_lines.append(
        f'{row["sha256"]}  {row["relative_path"]}'
    )
PHASE4I_SHA256SUMS = resolve_inside(
    PHASE4I_ROOT, "manifests/phase4i_SHA256SUMS.txt"
)
actual_sum_lines = [
    line.strip()
    for line in PHASE4I_SHA256SUMS.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]
assert actual_sum_lines == expected_sum_lines

write_json(
    AUDIT_DIR / "phase4j_cell01_authorization.json",
    {
        "audit_version": "phase4j_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "phase4i_manifest_rows": 69,
        "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
        "strict_deterministic": True,
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04I artifacts       : 69 / 69 VERIFIED")
print("Pseudo corpus             : VERIFIED")
print("Pseudo rows               : 113287")
print("Phase 04J authorization   : VERIFIED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Strict deterministic      : True")
print("Checkpoint payload read   : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 7 cells passed")
print("Next                      : production datasets and loaders")


PHASE 04J CELL 1 STATUS
Status                    : PASSED
Phase 04I artifacts       : 69 / 69 VERIFIED
Pseudo corpus             : VERIFIED
Pseudo rows               : 113287
Phase 04J authorization   : VERIFIED
GPU                       : Tesla T4
Strict deterministic      : True
Checkpoint payload read   : False
Full training started     : False
Test Set status           : SEALED
Progress                  : 1 / 7 cells passed
Next                      : production datasets and loaders


## Cell 2 — Human/pseudo datasets, stateless augmentation và DataLoaders

Đối chiếu pseudo corpus với đúng 113.287 Train+Restval image ID. Mỗi ảnh dùng
một human caption luân phiên theo epoch và một pseudo caption cố định.
Validation chỉ dùng human references; Held-out Test không được mở.


In [2]:
CELL_STARTED_AT = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
    filters=[("experiment_split", "in", ["train", "validation"])],
)
TRAIN_PATHS = {}
VALIDATION_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    image_path = leaf / Path(row["image_path"]).name
    target = (
        TRAIN_PATHS
        if row["experiment_split"] == "train"
        else VALIDATION_PATHS
    )
    target[image_id] = image_path
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_PATHS) == 5000

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append(
            (
                int(row["annotation_id"]),
                tuple(int(value) for value in row["token_ids"]),
            )
        )
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

PSEUDO_BY_IMAGE = {}
pseudo_file = pq.ParquetFile(PSEUDO_CORPUS)
assert pseudo_file.metadata.num_rows == 113287
assert set(pseudo_file.schema_arrow.names) == {
    "image_id", "teacher_caption", "student_token_ids",
    "student_lexical_tokens", "unknown_tokens", "truncated",
    "teacher_seed", "split",
}
for arrow_batch in pseudo_file.iter_batches(
    batch_size=16384,
    columns=[
        "image_id", "student_token_ids", "teacher_seed", "split"
    ],
):
    for row in arrow_batch.to_pylist():
        token_ids = tuple(int(value) for value in row["student_token_ids"])
        assert token_ids[0] == 1 and token_ids[-1] == 2
        assert len(token_ids) <= 32
        assert min(token_ids) >= 0 and max(token_ids) < 10080
        assert int(row["teacher_seed"]) == 42
        assert row["split"] == "Train+Restval"
        image_id = int(row["image_id"])
        assert image_id not in PSEUDO_BY_IMAGE
        PSEUDO_BY_IMAGE[image_id] = token_ids
assert set(PSEUDO_BY_IMAGE) == set(TRAIN_PATHS)

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for arrow_batch in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in arrow_batch.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append(
            (
                int(row["annotation_id"]),
                image_id,
                tuple(int(value) for value in row["token_ids"]),
            )
        )
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))


def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|phase4j"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)


def train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = (
            transforms.RandomResizedCrop.get_params(
                image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
            )
        )
        do_flip = bool(torch.rand(()) < 0.5)
    image = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        image = TF.hflip(image)
    return TF.to_tensor(image)


def validation_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    return TF.to_tensor(TF.center_crop(image, [224, 224]))


class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            image = train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
            "pseudo_ids": torch.tensor(
                PSEUDO_BY_IMAGE[image_id], dtype=torch.long
            ),
        }


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {"image_id": image_id, "student_image": image}


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(
            self.length, generator=generator
        ).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor


def pad_sequences(sequences):
    maximum = max(item.numel() for item in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, item in enumerate(sequences):
        result[index, :item.numel()] = item
    return result


def train_collate(rows):
    human = pad_sequences([row["human_ids"] for row in rows])
    pseudo = pad_sequences([row["pseudo_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "human_input_ids": human[:, :-1],
        "human_labels": human[:, 1:],
        "human_mask": human[:, :-1].ne(0),
        "pseudo_input_ids": pseudo[:, :-1],
        "pseudo_labels": pseudo[:, 1:],
        "pseudo_mask": pseudo[:, :-1].ne(0),
    }


def validation_caption_collate(rows):
    full = pad_sequences([row["token_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
        "decoder_attention_mask": full[:, :-1].ne(0),
    }


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }


TRAIN_DATASET = ProductionTrainDataset()
VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()


def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=32,
        sampler=EpochCursorSampler(
            len(TRAIN_DATASET), epoch=epoch, cursor=cursor
        ),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=train_collate,
    )


VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=validation_caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "student_images",
    "human_input_ids", "human_labels",
    "pseudo_input_ids", "pseudo_labels",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert len(set(batch_a["image_ids"].tolist())) == 32
assert 0.0 <= float(batch_a["student_images"].min())
assert float(batch_a["student_images"].max()) <= 1.0

write_json(
    AUDIT_DIR / "phase4j_cell02_data_pipeline.json",
    {
        "audit_version": "phase4j_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_images_per_epoch": len(TRAIN_DATASET),
        "human_captions_per_image_per_epoch": 1,
        "teacher_sequences_per_image_per_epoch": 1,
        "pseudo_membership_exact": True,
        "validation_captions": len(VALIDATION_CAPTION_DATASET),
        "validation_images": len(VALIDATION_IMAGE_DATASET),
        "physical_batch_size": 32,
        "normalization": "mean 0 std 1; tensor range [0,1]",
        "stateless_augmentation": True,
        "two_loader_exact": True,
        "resume_cursor_supported": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train images / epoch      :", len(TRAIN_DATASET))
print("Human captions / image    : 1, cyclic")
print("Teacher sequence / image  : 1, fixed")
print("Pseudo membership exact   : True")
print("Validation captions       :", len(VALIDATION_CAPTION_DATASET))
print("Validation images         :", len(VALIDATION_IMAGE_DATASET))
print("Physical/effective batch  : 32 / 64")
print("S2 input range            : [0,1]")
print("Two-loader exact          : True")
print("Resume cursor supported   : True")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 7 cells passed")
print("Next                      : S2 model and optimizer")


PHASE 04J CELL 2 STATUS
Status                    : PASSED
Train images / epoch      : 113287
Human captions / image    : 1, cyclic
Teacher sequence / image  : 1, fixed
Pseudo membership exact   : True
Validation captions       : 25010
Validation images         : 5000
Physical/effective batch  : 32 / 64
S2 input range            : [0,1]
Two-loader exact          : True
Resume cursor supported   : True
Test Set status           : SEALED
Progress                  : 2 / 7 cells passed
Next                      : S2 model and optimizer


## Cell 3 — S2 model, optimizer, scheduler và Sequence-KD step

Strict-load S2 encoder/decoder ban đầu. Teacher không nằm trong graph; pseudo
target được đọc từ corpus bất biến. Loss được chuẩn hóa riêng theo số token
human và pseudo để tránh caption dài chi phối.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion


PHASE2_ROOT = PHASE2_READY.parent
S2_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s2_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080


class S2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = timm.create_model(
            "mobilevit_s.cvnets_in1k", pretrained=False
        )
        if hasattr(backbone, "reset_classifier"):
            backbone.reset_classifier(num_classes=0, global_pool="")
        if hasattr(backbone, "head"):
            backbone.head = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Linear(640, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone.forward_features(images)
        if isinstance(feature, (tuple, list)):
            feature = [x for x in feature if torch.is_tensor(x)][-1]
        if isinstance(feature, dict):
            feature = [
                x for x in feature.values() if torch.is_tensor(x)
            ][-1]
        assert feature.shape[1:] == (640, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )


STUDENT_ENCODER = S2Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
with torch.serialization.safe_globals([TorchVersion]):
    encoder_payload = torch.load(
        S2_INITIAL_CHECKPOINT, map_location="cpu", weights_only=True
    )
    decoder_payload = torch.load(
        DECODER_INITIAL_CHECKPOINT,
        map_location="cpu", weights_only=True,
    )
STUDENT_ENCODER.load_state_dict(
    encoder_payload["state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    decoder_payload["state_dict"], strict=True
)
del encoder_payload, decoder_payload
assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 5114784
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328

STUDENT_ENCODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
STUDENT_ENCODER.to(device)
STUDENT_DECODER.to(device)


def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 32


def student_logits(batch):
    memory = STUDENT_ENCODER(batch["student_images"])
    return STUDENT_DECODER(
        batch["decoder_input_ids"],
        memory,
        batch["decoder_attention_mask"],
    )


ALPHA = 0.5
TEMPERATURE = None
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 2
STEPS_PER_EPOCH = math.ceil(113287 / 64)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def sequence_sum(logits, labels):
    return F.cross_entropy(
        logits.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=0,
        reduction="sum",
    )


def optimizer_step(
    microbatches, optimizer, scheduler, scaler, retry_limit=10
):
    human_total = sum(
        int(batch["human_labels"].ne(0).sum())
        for batch in microbatches
    )
    pseudo_total = sum(
        int(batch["pseudo_labels"].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0 and pseudo_total > 0
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_total = 0.0
        pseudo_sum_total = 0.0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = STUDENT_ENCODER(current["student_images"])
                human_logits = STUDENT_DECODER(
                    current["human_input_ids"],
                    memory,
                    current["human_mask"],
                )
                pseudo_logits = STUDENT_DECODER(
                    current["pseudo_input_ids"],
                    memory,
                    current["pseudo_mask"],
                )
                human_sum = sequence_sum(
                    human_logits, current["human_labels"]
                )
                pseudo_sum = sequence_sum(
                    pseudo_logits, current["pseudo_labels"]
                )
                normalized = (
                    (1 - ALPHA) * human_sum / human_total
                    + ALPHA * pseudo_sum / pseudo_total
                )
            scaler.scale(
                normalized / BACKWARD_DIVISOR
            ).backward()
            human_sum_total += float(human_sum)
            pseudo_sum_total += float(pseudo_sum)
        scaler.unscale_(optimizer)
        for parameter in (
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters())
        ):
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            1.0,
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        assert torch.isfinite(gradient_norm)
        scheduler.step()
        human_ce = human_sum_total / human_total
        pseudo_ce = pseudo_sum_total / pseudo_total
        return {
            "combined_loss": (
                (1 - ALPHA) * human_ce + ALPHA * pseudo_ce
            ),
            "human_ce": human_ce,
            "teacher_sequence_ce": pseudo_ce,
            "human_tokens": human_total,
            "teacher_tokens": pseudo_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771

write_json(
    CONFIG_DIR / "phase4j_production_optimizer_contract.json",
    {
        "contract_version": "phase4j_optimizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "optimizer": "AdamW",
        "learning_rates": {
            "backbone": 1e-4,
            "adapter": 3e-4,
            "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "maximum_epochs": MAX_EPOCHS,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16 plus GradScaler",
        "gradscaler_initial_scale": 1024.0,
        "overflow_retry_limit": 10,
        "backward_divisor": BACKWARD_DIVISOR,
        "gradient_clip": 1.0,
        "alpha": ALPHA,
        "temperature": None,
        "loss": "0.5 CE_human + 0.5 CE_teacher_sequence",
        "normalization": "each channel by its own valid-token count",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4j_cell03_model_optimizer.json",
    {
        "audit_version": "phase4j_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_loads": 2,
        "teacher_model_loaded": False,
        "pseudo_corpus_fixed": True,
        "student_parameters": 10874112,
        "optimizer_parameter_groups": 3,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "scheduler_advanced": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint loads   : 2 / 2 PASSED")
print("Teacher model loaded      : False")
print("Pseudo corpus fixed       : True")
print("S2 parameters             : 10874112")
print("Optimizer groups          : 3 / 3")
print("Physical/effective batch  : 32 / 64")
print("Steps / epoch             :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("FP16 / GradScaler         : ENABLED / 1024")
print("Overflow retry limit      : 10")
print("Combined loss             : 0.5 human + 0.5 teacher sequence")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 7 cells passed")
print("Next                      : Validation and Beam-3 engine")


PHASE 04J CELL 3 STATUS
Status                    : PASSED
Strict checkpoint loads   : 2 / 2 PASSED
Teacher model loaded      : False
Pseudo corpus fixed       : True
S2 parameters             : 10874112
Optimizer groups          : 3 / 3
Physical/effective batch  : 32 / 64
Steps / epoch             : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
FP16 / GradScaler         : ENABLED / 1024
Overflow retry limit      : 10
Combined loss             : 0.5 human + 0.5 teacher sequence
Optimizer step performed  : False
Test Set status           : SEALED
Progress                  : 3 / 7 cells passed
Next                      : Validation and Beam-3 engine


## Cell 4 — Offline COCO metrics, teacher-forced Validation và Beam-3

Kiểm tra backend metric, CE Validation và generation lặp lại chính xác.
Checkpoint selection tiếp tục dùng CIDEr-D trên Validation.


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "phase4j_cell04_validation_engine.json",
    {
        "audit_version": "phase4j_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : atomic checkpoint and production pilot")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 24679.40 tokens per second.
PTBTokenizer tokenized 385 tokens at 5355.34 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
PHASE 04J CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 11.258756868131869
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : atomic checkpoint and production pilot


## Cell 5 — Atomic checkpoint và discarded production pilot

Chạy đúng 4 optimizer-step trên 256 ảnh, kiểm tra loss/gradient/overflow,
save-load chính xác rồi xóa checkpoint và phục hồi trọng số ban đầu.


In [5]:
CELL_STARTED_AT = time.perf_counter()


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if isinstance(left, torch.Tensor):
        return (
            isinstance(right, torch.Tensor)
            and left.dtype == right.dtype
            and left.shape == right.shape
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(nested_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(nested_exact(a, b) for a, b in zip(left, right))
        )
    return left == right


def restore_student_initial():
    STUDENT_ENCODER.load_state_dict(
        STUDENT_ENCODER_INITIAL, strict=True
    )
    STUDENT_DECODER.load_state_dict(
        STUDENT_DECODER_INITIAL, strict=True
    )
    for module in [STUDENT_ENCODER, STUDENT_DECODER]:
        module.zero_grad(set_to_none=True)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_training_checkpoint(
    path, optimizer, scheduler, scaler, progress
):
    atomic_save(
        path,
        {
            "schema_version": (
                "phase4_teacher_b_s2_sequence_kd_checkpoint_v1"
            ),
            "experiment_id": "P4B_B_S2_SEQKD",
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
            "alpha": ALPHA,
            "temperature": None,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


restore_student_initial()
optimizer, scheduler, scaler = build_optimizer_state()
pilot_iterator = iter(build_train_loader(epoch=0, cursor=0))
pilot_rows = []
pilot_started = time.perf_counter()
cursor = 0
overflow_total = 0
for optimizer_step_index in range(1, 5):
    microbatches = [
        next(pilot_iterator) for _ in range(ACCUMULATION)
    ]
    result = optimizer_step(
        microbatches, optimizer, scheduler, scaler,
        retry_limit=10,
    )
    cursor += sum(
        len(batch["image_ids"]) for batch in microbatches
    )
    overflow_total += result["overflow_retries"]
    row = {
        "optimizer_step": optimizer_step_index,
        "cursor": cursor,
        **result,
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_adapter": optimizer.param_groups[1]["lr"],
        "lr_decoder": optimizer.param_groups[2]["lr"],
    }
    pilot_rows.append(row)
    print(
        f"step={optimizer_step_index:02d} | cursor={cursor} | "
        f"loss={result['combined_loss']:.6f} | "
        f"human={result['human_ce']:.6f} | "
        f"pseudo={result['teacher_sequence_ce']:.6f} | "
        f"grad={result['gradient_norm']:.4f}"
    )
pilot_seconds = time.perf_counter() - pilot_started
assert cursor == 256
assert all(
    math.isfinite(row["combined_loss"]) for row in pilot_rows
)

TEMP_CHECKPOINT = CHECKPOINT_DIR / "temporary_production_pilot.pt"
progress = {
    "epoch": 0,
    "cursor": cursor,
    "global_optimizer_step": 4,
}
save_training_checkpoint(
    TEMP_CHECKPOINT, optimizer, scheduler, scaler, progress
)
expected_encoder = clone_cpu(STUDENT_ENCODER.state_dict())
expected_decoder = clone_cpu(STUDENT_DECODER.state_dict())
expected_optimizer = clone_cpu(optimizer.state_dict())
expected_scheduler = clone_cpu(scheduler.state_dict())
expected_scaler = clone_cpu(scaler.state_dict())
expected_cpu_rng = torch.get_rng_state().clone()
expected_cuda_rng = [
    state.clone() for state in torch.cuda.get_rng_state_all()
]

restore_student_initial()
loaded_optimizer, loaded_scheduler, loaded_scaler = (
    build_optimizer_state()
)
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        TEMP_CHECKPOINT, map_location="cpu", weights_only=True
    )
STUDENT_ENCODER.load_state_dict(
    checkpoint["student_encoder_state"], strict=True
)
STUDENT_DECODER.load_state_dict(
    checkpoint["student_decoder_state"], strict=True
)
loaded_optimizer.load_state_dict(checkpoint["optimizer_state"])
loaded_scheduler.load_state_dict(checkpoint["scheduler_state"])
loaded_scaler.load_state_dict(checkpoint["scaler_state"])
torch.set_rng_state(checkpoint["cpu_rng"])
torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])

assert nested_exact(expected_encoder, STUDENT_ENCODER.state_dict())
assert nested_exact(expected_decoder, STUDENT_DECODER.state_dict())
assert nested_exact(expected_optimizer, loaded_optimizer.state_dict())
assert nested_exact(expected_scheduler, loaded_scheduler.state_dict())
assert nested_exact(expected_scaler, loaded_scaler.state_dict())
assert torch.equal(expected_cpu_rng, torch.get_rng_state())
assert all(
    torch.equal(left, right)
    for left, right in zip(
        expected_cuda_rng, torch.cuda.get_rng_state_all()
    )
)
TEMP_CHECKPOINT.unlink()
assert not TEMP_CHECKPOINT.exists()
restore_student_initial()

PILOT_CURVE = LOG_DIR / "phase4j_discarded_pilot.csv"
with PILOT_CURVE.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(pilot_rows[0])
    )
    writer.writeheader()
    writer.writerows(pilot_rows)

seconds_per_step = pilot_seconds / 4
estimated_minutes_per_epoch = (
    seconds_per_step * STEPS_PER_EPOCH / 60
)
write_json(
    AUDIT_DIR / "phase4j_cell05_production_pilot.json",
    {
        "audit_version": "phase4j_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_steps": 4,
        "microbatches": 8,
        "images": cursor,
        "overflow_retries": overflow_total,
        "seconds_per_optimizer_step": seconds_per_step,
        "estimated_minutes_per_epoch": estimated_minutes_per_epoch,
        "atomic_checkpoint": True,
        "resume_load_exact": True,
        "temporary_checkpoint_deleted": True,
        "student_initial_restore_exact": True,
        "pseudo_corpus_fixed": True,
        "pilot_weights_saved": False,
        "full_validation_run": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Pilot optimizer steps     : 4 / 4")
print("Pilot images              :", cursor)
print("Overflow retries          :", overflow_total)
print("Seconds / optimizer step  :", round(seconds_per_step, 4))
print("Estimated min / epoch     :", round(estimated_minutes_per_epoch, 2))
print("Atomic checkpoint         : PASSED")
print("Resume load exact         : True")
print("Temporary checkpoint      : DELETED")
print("Initial restore exact     : True")
print("Pilot weights saved       : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 7 cells passed")
print("Next                      : lock production contract")


/tmp/ipykernel_23/3471408732.py:278: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  human_sum_total += float(human_sum)


step=01 | cursor=64 | loss=11.013464 | human=11.016911 | pseudo=11.010018 | grad=6.5677
step=02 | cursor=128 | loss=11.038691 | human=11.113673 | pseudo=10.963708 | grad=6.6574
step=03 | cursor=192 | loss=11.060482 | human=11.065387 | pseudo=11.055577 | grad=6.5711
step=04 | cursor=256 | loss=10.990004 | human=10.994575 | pseudo=10.985434 | grad=6.5859
PHASE 04J CELL 5 STATUS
Status                    : PASSED
Pilot optimizer steps     : 4 / 4
Pilot images              : 256
Overflow retries          : 0
Seconds / optimizer step  : 1.6317
Estimated min / epoch     : 48.16
Atomic checkpoint         : PASSED
Resume load exact         : True
Temporary checkpoint      : DELETED
Initial restore exact     : True
Pilot weights saved       : False
Full training started     : False
Test Set status           : SEALED
Progress                  : 5 / 7 cells passed
Next                      : lock production contract


## Cell 6 — Lock production run contract

Khóa loss, pseudo corpus, Student initialization, optimizer, Validation,
checkpoint selection, early stopping và resume boundary.


In [6]:
CELL_STARTED_AT = time.perf_counter()
PRODUCTION_CONTRACT = (
    CONFIG_DIR / "phase4j_teacher_b_s2_sequence_kd_production_contract.json"
)
production_contract = {
    "contract_version": "phase4j_teacher_b_s2_sequence_kd_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": "P4B_B_S2_SEQKD",
    "teacher": {
        "id": "B",
        "seed": 42,
        "model_loaded_during_student_training": False,
        "knowledge_channel": "fixed pseudo-caption sequence",
        "pseudo_corpus": str(PSEUDO_CORPUS),
        "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
        "pseudo_rows": 113287,
    },
    "student": {
        "id": "S2",
        "seed": 42,
        "initial_encoder_checkpoint": str(S2_INITIAL_CHECKPOINT),
        "initial_decoder_checkpoint": str(
            DECODER_INITIAL_CHECKPOINT
        ),
    },
    "kd": {
        "channel": "sequence_pseudo_caption",
        "loss": "0.5 CE_human + 0.5 CE_teacher_sequence",
        "alpha": ALPHA,
        "temperature": None,
        "human_and_teacher_denominators": "separate valid-token counts",
        "direct_teacher_logits_used": False,
    },
    "training": {
        "images_per_epoch": 113287,
        "human_caption_sampling": "one per image per epoch cyclic",
        "teacher_caption_sampling": "one fixed sequence per image",
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "maximum_epochs": 20,
        "optimizer": "AdamW",
        "scheduler": "warmup 5 percent plus cosine",
        "mixed_precision": "FP16 plus GradScaler",
        "gradient_clip": 1.0,
        "overflow_retry_limit": 10,
        "atomic_last_every_optimizer_steps": 250,
        "resume_boundary": "optimizer step",
    },
    "validation": {
        "caption_count": 25010,
        "image_count": 5000,
        "generation": "Beam-3 length penalty 0.7",
        "metrics": [
            "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"
        ],
        "selection_primary": "CIDEr-D",
        "tie_break": ["BLEU-4", "CE", "earlier epoch"],
    },
    "early_stopping": {
        "minimum_epoch": 5,
        "patience": 4,
        "minimum_delta": 0.001,
    },
    "comparison": {
        "baseline": "S2 CE-only seed42",
        "paired_seed": 42,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PRODUCTION_CONTRACT, production_contract)
write_json(
    AUDIT_DIR / "phase4j_cell06_production_contract.json",
    {
        "audit_version": "phase4j_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "contract": str(PRODUCTION_CONTRACT),
        "contract_sha256": sha256_file(PRODUCTION_CONTRACT),
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "atomic_last_checkpoint": True,
        "atomic_best_checkpoint": True,
        "mid_epoch_resume": True,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04J CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Production contract       : LOCKED")
print("Maximum epochs            : 20")
print("Pseudo corpus             : LOCKED")
print("Student initialization    : LOCKED")
print("Selection split           : Validation")
print("Primary metric            : CIDEr-D")
print("Beam size / penalty       : 3 / 0.7")
print("Atomic last / best        : READY / READY")
print("Mid-epoch resume          : READY")
print("Early stopping            : epoch>=5 / patience=4")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 7 cells passed")
print("Next                      : package and authorize Phase 04K")


PHASE 04J CELL 6 STATUS
Status                    : PASSED
Production contract       : LOCKED
Maximum epochs            : 20
Pseudo corpus             : LOCKED
Student initialization    : LOCKED
Selection split           : Validation
Primary metric            : CIDEr-D
Beam size / penalty       : 3 / 0.7
Atomic last / best        : READY / READY
Mid-epoch resume          : READY
Early stopping            : epoch>=5 / patience=4
Full training started     : False
Test Set status           : SEALED
Progress                  : 6 / 7 cells passed
Next                      : package and authorize Phase 04K


## Cell 7 — Package Phase 04J và authorize Phase 04K

Tạo manifest CSV/JSON/SHA256SUMS, marker COMPLETE và authorization cho
Teacher B → S2 Sequence-KD full run seed 42.


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4j_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4j_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4j_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "TRAINING_READINESS_COMPLETE.json"
)
AUTHORIZED_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "SEED42_FULL_RUN_AUTHORIZED.json"
)

write_json(
    AUTHORIZED_MARKER,
    {
        "marker_version": (
            "phase4_teacher_b_s2_sequence_kd_seed42_"
            "full_run_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4B_B_S2_SEQKD",
        "seed": 42,
        "production_contract": {
            "relative_path": PRODUCTION_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PRODUCTION_CONTRACT),
        },
        "pseudo_corpus": {
            "source_sha256": sha256_file(PSEUDO_CORPUS),
            "rows": 113287,
        },
        "next_phase": (
            "Phase 04K Teacher B to S2 Sequence-KD seed42 full run"
        ),
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, COMPLETE_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4j_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase4_teacher_b_s2_sequence_kd_"
            "training_readiness_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": "P4B_B_S2_SEQKD",
        "seed": 42,
        "production_data_pipeline": "passed",
        "model_optimizer_engine": "passed",
        "validation_beam_engine": "passed",
        "discarded_pilot": "passed",
        "production_contract": "locked",
        "phase04k_full_run_authorized": True,
        "authorization_marker": {
            "relative_path": AUTHORIZED_MARKER.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AUTHORIZED_MARKER),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "pilot_optimizer_steps_only": True,
        "pilot_weights_saved": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04J CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Experiment                : Teacher B -> S2 Sequence-KD")
print("Production data pipeline  : PASSED")
print("Model/optimizer engine    : PASSED")
print("Validation/Beam-3 engine  : PASSED")
print("Discarded pilot           : 4 / 4 PASSED")
print("Production contract       : LOCKED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 04K full seed42 run : AUTHORIZED")
print("Pilot weights saved       : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Progress                  : 7 / 7 cells passed")
print("Next                      : Save Version, then Phase 04K")


PHASE 04J CELL 7 STATUS
Status                    : COMPLETE
Experiment                : Teacher B -> S2 Sequence-KD
Production data pipeline  : PASSED
Model/optimizer engine    : PASSED
Validation/Beam-3 engine  : PASSED
Discarded pilot           : 4 / 4 PASSED
Production contract       : LOCKED
Manifest rows             : 11
Manifest verification     : ALL PASSED
Phase 04K full seed42 run : AUTHORIZED
Pilot weights saved       : False
Full training started     : False
Test Set status           : SEALED
Elapsed                   : 0.03 giây
Progress                  : 7 / 7 cells passed
Next                      : Save Version, then Phase 04K
